# HRSA — Delaware peek (Q1 underserved layer)

Scope: HRSA shortage/underservice designations for DE. HPSA Primary Care + MUA/P.
Source: `data/sources/hrsa/HPSA_Primary_Care.csv`, `data/sources/hrsa/MUA.csv`.
Reading: read whole (small), filter to DE by state abbreviation, active designations only.

In [1]:
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
HPSA = ROOT / "data/sources/hrsa/HPSA_Primary_Care.csv"
hp = pd.read_csv(HPSA, dtype=str, encoding="latin-1", low_memory=False)
de = hp[hp["Primary State Abbreviation"] == "DE"].copy()
des = de[de["HPSA Status"] == "Designated"]
print("HPSA PC: all", hp.shape, "| DE", de.shape, "| DE Designated", des.shape)
des[["HPSA Name", "Designation Type", "HPSA Score", "Common County Name", "HPSA Estimated Underserved Population"]].head(10)

HPSA PC: all (79750, 66) | DE (315, 66) | DE Designated (88, 66)


,HPSA Name,Designation Type,HPSA Score,Common County Name,HPSA Estimated Underserved Population
63610,"La Red Health Center, Inc.",Federally Qualified Health Center,21,"Sussex County, DE",NaN
63612,WESTSIDE FAMILY HEALTHCARE INC,Federally Qualified Health Center,21,"New Castle County, DE",NaN
63613,"SOUTHBRIDGE MEDICAL ADVISORY COUNCIL, INC.",Federally Qualified Health Center,20,"New Castle County, DE",NaN
63641,CF-Howard R. Young Correctional Institution,Correctional Facility,3,"New Castle County, DE",NaN
63642,CF-Sussex Correctional Institution,Correctional Facility,3,"Sussex County, DE",NaN
64582,Sussex Correctional Institution,Correctional Facility,6,"Sussex County, DE",NaN
64584,ME - Sussex County,HPSA Population,14,"Sussex County, DE",118694
65446,LI - Wilmington/Stanton/Newark,HPSA Population,16,"New Castle County, DE",81861
65447,LI - Wilmington/Stanton/Newark,HPSA Population,16,"New Castle County, DE",81861
65448,LI - Wilmington/Stanton/Newark,HPSA Population,16,"New Castle County, DE",81861


In [2]:
# DE HPSA: status, designation type, score range, county spread.
print("DE status:\n", de["HPSA Status"].value_counts().to_string())
print("\nDE Designated by type:\n", des["Designation Type"].value_counts().to_string())
sc = pd.to_numeric(des["HPSA Score"], errors="coerce")
print("\nDesignated HPSA Score -> min:", sc.min(), "max:", sc.max(), "mean:", round(sc.mean(), 1))
print("\nDesignated by county:\n", des["Common County Name"].value_counts().to_string())

DE status:
 HPSA Status
Withdrawn                  149
Designated                  88
Proposed For Withdrawal     78

DE Designated by type:
 Designation Type
HPSA Population                      79
Correctional Facility                 6
Federally Qualified Health Center     3

Designated HPSA Score -> min: 3 max: 21 mean: 15.3

Designated by county:
 Common County Name
New Castle County, DE    83
Sussex County, DE         4
Kent County, DE           1


In [3]:
# MUA/P for DE (area/tract-based underservice).
MUA = ROOT / "data/sources/hrsa/MUA.csv"
mu = pd.read_csv(MUA, dtype=str, encoding="latin-1", low_memory=False)
mde = mu[mu["State Abbreviation"] == "DE"].copy()
print("MUA: all", mu.shape, "| DE", mde.shape)
print("\nDE status:\n", mde["MUA/P Status Description"].value_counts().to_string())
imu = pd.to_numeric(mde["IMU Score"], errors="coerce")
print("\nIMU score -> min:", imu.min(), "max:", imu.max())
mde[["MUA/P Service Area Name", "Designation Type", "MUA/P Status Description", "IMU Score", "County Equivalent Name", "Census Tract"]].head(10)

MUA: all (20132, 63) | DE (24, 63)

DE status:
 MUA/P Status Description
Designated    24

IMU score -> min: 57.2 max: 90.4


,MUA/P Service Area Name,Designation Type,MUA/P Status Description,IMU Score,County Equivalent Name,Census Tract
558,New Castle Service Area,Medically Underserved Area,Designated,60.10,New Castle,001902
559,New Castle Service Area,Medically Underserved Area,Designated,59.80,New Castle,000400
560,New Castle Service Area,Medically Underserved Area,Designated,59.80,New Castle,002800
561,New Castle Service Area,Medically Underserved Area,Designated,59.80,New Castle,001100
562,New Castle Service Area,Medically Underserved Area,Designated,59.80,New Castle,000900
563,New Castle Service Area,Medically Underserved Area,Designated,59.80,New Castle,003002
564,New Castle County,Medically Underserved Area,Designated,57.20,New Castle,001500
565,New Castle County,Medically Underserved Area,Designated,57.20,New Castle,001600
566,New Castle County,Medically Underserved Area,Designated,57.20,New Castle,001400
567,New Castle County,Medically Underserved Area,Designated,57.20,New Castle,002100


## Notes
- Q1 underserved layer. Filter DE by state abbreviation; use active designations only (`HPSA Status == "Designated"`).
- HPSA Score 1-25 (higher = worse shortage; NHSC assignment priority). IMU <= 62 = underserved.
- HPSA designation types differ: geographic (area) vs population-group vs facility (FQHC / correctional). Use geographic + population for area maps; facility rows are point sites.
- Join to providers / counties by FIPS or county name. Definitions: `docs/data-dictionaries/hrsa.md`.